[article][1]

[1]: https://www.geeksforgeeks.org/deep-learning/contrastive-learning-with-simclr-in-pytorch/

# 1. Setup: Install Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
import torchvision.models as models
import numpy as np
import random
from tqdm import tqdm

# 2. Data Augmentation


In [2]:
simclr_transform = T.Compose([
    T.RandomResizedCrop(32),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# 3. Dataset for Two Views


In [3]:
from SimCLRDataset import SimCLRDataset

# 4. SimCLR Model = Encoder + Projection Head


In [4]:
class SimCLRModel(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        base_model = models.resnet18(weights=None)
        num_ftrs = base_model.fc.in_features
        base_model.fc = nn.Identity()
        self.encoder = base_model
        self.projection_head = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

# 5. NT-Xent Loss


In [5]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)

    similarity = torch.matmul(z, z.T)
    N = z_i.shape[0]

    mask = (~torch.eye(2*N, dtype=bool)).to(z.device)
    sim = similarity / temperature
    exp_sim = torch.exp(sim) * mask

    positive_sim = torch.exp(F.cosine_similarity(z_i, z_j) / temperature)
    positives = torch.cat([positive_sim, positive_sim], dim=0)

    denominator = exp_sim.sum(dim=1)
    loss = -torch.log(positives / denominator)
    return loss.mean()

# 6. Training Loop


In [6]:
# 6.1
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [7]:
# 6.2
train_dataset = CIFAR10(root='./data', train=True, download=True)
contrastive_dataset = SimCLRDataset(train_dataset, simclr_transform)
train_loader = DataLoader(contrastive_dataset, batch_size=256, shuffle=True, num_workers=2)


In [8]:
%%time 
# 6.3
model = SimCLRModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

for epoch in range(10):
    model.train()
    total_loss = 0
    for x_i, x_j in tqdm(train_loader):
        x_i, x_j = x_i.to(device), x_j.to(device)
        z_i = model(x_i)
        z_j = model(x_j)

        loss = nt_xent_loss(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:24<00:00,  8.07it/s]


Epoch 1 | Loss: 5.5193


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:24<00:00,  8.02it/s]


Epoch 2 | Loss: 5.3405


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:24<00:00,  8.07it/s]


...

Epoch 8 | Loss: 5.1707


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:24<00:00,  7.89it/s]


Epoch 9 | Loss: 5.1543


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:30<00:00,  6.46it/s]

Epoch 10 | Loss: 5.1447
CPU times: user 41.3 s, sys: 4.79 s, total: 46.1 s
Wall time: 4min 9s


# 7. Evaluation (Linear Probe)

In [9]:
for param in model.encoder.parameters():
    param.requires_grad = False

classifier = nn.Linear(512, 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

def get_features_and_labels(loader):
    features, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            h = model.encoder(x)
            features.append(h.cpu())
            labels.append(y)
    return torch.cat(features), torch.cat(labels)

In [10]:
# Create test dataset and data loader
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

test_features, test_labels = get_features_and_labels(test_loader)
test_dataset_feat = torch.utils.data.TensorDataset(test_features, test_labels)
test_loader_feat = torch.utils.data.DataLoader(test_dataset_feat, batch_size=256, shuffle=False)


In [11]:

accuracy = evaluate(classifier, test_loader_feat)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

NameError: name 'evaluate' is not defined